Direct download pattern:

https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YEAR}_{MONTH}.zip

e.g.:

bash
wget --no-check-certificate https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

Each zip is one month, contains all carriers and all airports for that month already — no per-carrier/per-airport looping needed at all, that's only relevant if you use the interactive web UI at Departures.aspx, which is the wrong entry point for bulk downloads. Just loop YEAR (1987–present) × MONTH (1–12), which is a clean, trivial Airflow DAG:

python
import requests

BASE = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"

for year in range(2018, 2026):
    for month in range(1, 13):
        url = BASE.format(year=year, month=month)
        # download, unzip, load into GCS/staging

Each zip contains one CSV (the schema is the same as what you'd get from the interactive download builder — flight date, carrier, origin, dest, scheduled/actual times, delay-cause minutes, cancellations).

Worth noting for your DAG: BTS updates with roughly a 1–2 month lag, so don't request the most recent 1-2 months blindly — check what's actually published first, or your DAG will just get 404s for months that don't exist yet.

If you also want the DB1B ticket/itinerary survey (10% sample of actual tickets — origin/destination/fare, closer to what you worked with at Kiwi.com) rather than just delay stats, that's a separate PREZIP pattern:

https://transtats.bts.gov/PREZIP/Origin_and_Destination_Survey_DB1BCoupon_{year}_{quarter}.zip

quarterly, not monthly, 1993–present.

In [4]:
!wget --no-check-certificate -O data/tmp.zip https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

--2026-08-10 19:43:07--  https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip
Resolving transtats.bts.gov (transtats.bts.gov)... 204.68.194.70
Connecting to transtats.bts.gov (transtats.bts.gov)|204.68.194.70|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27573265 (26M) [application/x-zip-compressed]
Saving to: ‘data/tmp.zip’

data/tmp.zip        100%[===================>]  26.30M   305KB/s    in 1m 50s  

2026-08-10 19:44:58 (245 KB/s) - ‘data/tmp.zip’ saved [27573265/27573265]



In [6]:
!unzip -o data/tmp.zip -d ./data/extracted/

Archive:  data/tmp.zip
  inflating: ./data/extracted//On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv  
  inflating: ./data/extracted//readme.html  


In [2]:
import pandas as pd

In [5]:
!pwd

/Users/filip/Developer/us-flight-delays-pipeline/notebooks


In [6]:
df = pd.read_csv('/Users/filip/Developer/us-flight-delays-pipeline/data/extracted/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv')

/var/folders/05/3xzngw_n6tv7wlp__kmhfjwh0000gn/T/ipykernel_65070/3716885186.py:1: DtypeWarning: Columns (0: Div2Airport, 1: Div2TailNum, 2: Div3Airport, 3: Div3TailNum) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/filip/Developer/us-flight-delays-pipeline/data/extracted/On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv')


In [7]:
df.describe()

,Year,Quarter,Month,DayofMonth,DayOfWeek,DOT_ID_Reporting_Airline,Flight_Number_Reporting_Airline,OriginAirportID,OriginAirportSeqID,OriginCityMarketID,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
count,547271.0,547271.0,547271.0,547271.000000,547271.000000,547271.000000,547271.000000,547271.000000,5.472710e+05,547271.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,2024.0,1.0,1.0,15.893364,3.802931,19943.650341,2344.884295,12659.022795,1.265906e+06,31751.846465,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.0,0.0,0.0,8.954236,2.012839,373.913358,1576.254272,1526.276446,1.526274e+05,1320.563507,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2024.0,1.0,1.0,1.000000,1.000000,19393.000000,1.000000,10135.000000,1.013506e+06,30070.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2024.0,1.0,1.0,8.000000,2.000000,19790.000000,1083.000000,11292.000000,1.129202e+06,30647.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2024.0,1.0,1.0,16.000000,4.000000,19805.000000,2069.000000,12889.000000,1.288904e+06,31454.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2024.0,1.0,1.0,24.000000,6.000000,20363.000000,3454.000000,14027.000000,1.402702e+06,32467.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2024.0,1.0,1.0,31.000000,7.000000,20452.000000,8819.000000,16869.000000,1.686902e+06,35991.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
for column in df.columns: print(column)

Year
Quarter
Month
DayofMonth
DayOfWeek
FlightDate
Reporting_Airline
DOT_ID_Reporting_Airline
IATA_CODE_Reporting_Airline
Tail_Number
Flight_Number_Reporting_Airline
OriginAirportID
OriginAirportSeqID
OriginCityMarketID
Origin
OriginCityName
OriginState
OriginStateFips
OriginStateName
OriginWac
DestAirportID
DestAirportSeqID
DestCityMarketID
Dest
DestCityName
DestState
DestStateFips
DestStateName
DestWac
CRSDepTime
DepTime
DepDelay
DepDelayMinutes
DepDel15
DepartureDelayGroups
DepTimeBlk
TaxiOut
WheelsOff
WheelsOn
TaxiIn
CRSArrTime
ArrTime
ArrDelay
ArrDelayMinutes
ArrDel15
ArrivalDelayGroups
ArrTimeBlk
Cancelled
CancellationCode
Diverted
CRSElapsedTime
ActualElapsedTime
AirTime
Flights
Distance
DistanceGroup
CarrierDelay
WeatherDelay
NASDelay
SecurityDelay
LateAircraftDelay
FirstDepTime
TotalAddGTime
LongestAddGTime
DivAirportLandings
DivReachedDest
DivActualElapsedTime
DivArrDelay
DivDistance
Div1Airport
Div1AirportID
Div1AirportSeqID
Div1WheelsOn
Div1TotalGTime
Div1LongestGTime
Div1W

In [10]:
df.shape

(547271, 110)

In [12]:
null_summary = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('null_pct', ascending=False)

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(null_summary)


,null_count,null_pct
Unnamed: 109,547271,100.00
Div4AirportSeqID,547271,100.00
Div2WheelsOff,547250,100.00
Div2TailNum,547250,100.00
Div3Airport,547270,100.00
Div3AirportID,547270,100.00
Div3AirportSeqID,547270,100.00
Div3WheelsOn,547270,100.00
Div3TotalGTime,547270,100.00
Div3LongestGTime,547270,100.00


# Duplication check

to avoid duplicating delayed flights reported carrier codeshare Partner

In [15]:
dupe_check = (
    df.groupby(['FlightDate', 'Origin', 'Dest', 'CRSDepTime'])
      .agg(
          n_rows=('Reporting_Airline', 'size'),
          n_distinct_carriers=('Reporting_Airline', 'nunique'),
          carriers=('Reporting_Airline', lambda s: sorted(s.unique())),
      )
      .reset_index()
)

likely_codeshares = dupe_check[dupe_check['n_distinct_carriers'] > 1]
print(f"{len(likely_codeshares):,} scheduled slots reported by >1 carrier code "
      f"out of {len(dupe_check):,} ({len(likely_codeshares) / len(dupe_check) * 100:.2f}%)")

likely_codeshares.sort_values('n_rows', ascending=False).head(10)

3,439 scheduled slots reported by >1 carrier code out of 543,597 (0.63%)


,FlightDate,Origin,Dest,CRSDepTime,n_rows,n_distinct_carriers,carriers
274294,2024-01-16,LAS,SEA,600,4,4,"[AS, DL, F9, NK]"
205489,2024-01-12,LAS,SEA,600,4,4,"[AS, DL, F9, NK]"
406653,2024-01-24,BOS,LGA,600,3,3,"[AA, B6, DL]"
501934,2024-01-29,LAS,SEA,600,3,3,"[AS, DL, NK]"
264029,2024-01-15,SFO,LAX,600,3,3,"[DL, OO, UA]"
136402,2024-01-08,LGA,ORD,600,3,3,"[AA, DL, UA]"
403179,2024-01-23,SFO,LAX,600,3,3,"[DL, OO, UA]"
173206,2024-01-10,ORD,LGA,600,3,3,"[AA, DL, UA]"
315711,2024-01-18,SFO,LAX,600,3,3,"[DL, OO, UA]"
438188,2024-01-25,SFO,LAX,600,3,3,"[DL, OO, UA]"


- Lower than expected, but I am use to checking international flights. True is that domestic flights are run lowcosts and they never codeshare.
- Even results can be just coicidence for frequent routes or even competitive behaviour.
- Noticed we have tail number available so this is true aircraft designator, se with this we can easy check if flight is the same.

In [14]:
physical_flight_key = ['FlightDate', 'Tail_Number', 'Origin', 'Dest', 'CRSDepTime']

dupe_groups = (
    df[df['Tail_Number'].notna()]  # can't verify without a tail number
      .groupby(physical_flight_key)
      .agg(n_rows=('Reporting_Airline', 'size'),
           carriers=('Reporting_Airline', lambda s: sorted(s.unique())))
      .reset_index()
)

true_dupes = dupe_groups[dupe_groups['n_rows'] > 1]
print(f"Confirmed same-aircraft duplicate filings: {len(true_dupes):,} "
      f"out of {len(dupe_groups):,} physical flights "
      f"({len(true_dupes) / len(dupe_groups) * 100:.3f}%)")

Confirmed same-aircraft duplicate filings: 0 out of 541,978 physical flights (0.000%)


Clean result, no dedup step needed.
Reporting_Airline cleanly means "the carrier that operated this flight" — so your delay-probability-by-carrier breakdown can honestly be framed as operational reliability (who actually flew the plane), not just "whose code was on the ticket.
Worth documenting!

## Column reduction & data quality decisions

The raw `On_Time_Reporting_Carrier_On_Time_Performance` extract has **109 columns**, most of which are either junk, near-empty diversion detail, or redundant identifiers. Before designing the staging schema, we investigated the data rather than dropping columns blindly — findings and decisions below.

### 1. Junk columns — drop outright

| Column | Reason |
|---|---|
| `Unnamed: 109` | Phantom column from a trailing comma in every row (BTS export quirk). 100% null, not real data. |

### 2. Diversion detail (`Div1`–`Div5`, 40 columns) — collapse, don't carry forward

Initially assumed these were connecting-flight segments; confirmed they're not. A diversion is a **single scheduled flight** (one flight number, one row) that got unexpectedly rerouted mid-flight (weather, mechanical, medical). `Origin`/`Dest` stay as originally scheduled; `Div1Airport`...`Div5Airport` record where it actually went if it kept getting rerouted further. 96–100% null across the board since diversions are rare.

**Decision:** don't carry 40 sparse wide columns into the fact table. Instead:
- `fact_flight_leg` gets two summary columns: **`was_diverted`** (boolean, from `Diverted`) and **`final_destination`** (resolves to the last non-null `DivN Airport` if diverted, else `Dest`).
- Full diversion detail (per-diversion-leg ground time, wheels on/off, tail number at each stop) moves to a separate **`diversion_detail`** table, one row per `(flight_id, diversion_sequence 1–5)`, built by unpivoting the `Div1`–`Div5` blocks with `UNION ALL`. Keeps the fact table narrow; detail is still queryable for anyone who actually needs it.

### 3. Delay-cause columns (`CarrierDelay`, `WeatherDelay`, `NASDelay`, `SecurityDelay`, `LateAircraftDelay`) — keep, document sparsity

~77% null, but **not a data quality problem** — BTS only populates these when a flight is delayed ≥15 minutes, per reporting methodology. This is core to the delay-cause analysis the dashboard is built around, so these stay as-is in the fact table with sparsity documented rather than "fixed."

### 4. Codeshare duplication — investigated, confirmed not present in this dataset

Since Jan 1, 2018, US DOT (14 CFR §234.4(k)) requires marketing carriers to separately file on-time performance for codeshare flights operated by a partner — meaning the *same physical flight* can legitimately appear as multiple rows under different carrier codes.

Tested for this directly using the true physical-flight fingerprint — `(FlightDate, Tail_Number, Origin, Dest, CRSDepTime)` — rather than naively checking for duplicate origin/destination per day (which false-positives on routes flown multiple times daily) or by scheduled time alone (which false-positives on unrelated flights coincidentally departing the same route at the same popular slot, e.g. 6:00am shuttle departures).

**Result: 0 confirmed same-aircraft duplicate filings out of 541,978 physical flights.** Regional operators (SkyWest `OO`, Envoy `MQ`, Republic `YX`, etc.) already meet BTS's independent reporting threshold and file under their own carrier code, so the §234.4(k) "fill the gap" requirement rarely triggers in this dataset. Full methodology in `docs/data_quality_notes.md`.

**Implication:** no deduplication step needed in staging. `Reporting_Airline` can be treated as the *operating* carrier for delay-probability-by-carrier analysis — a stronger, more honest claim than most flight-delay dashboards can make, since we verified it rather than assumed it.

### 5. Fully-populated core columns — keep as-is

Date/time fields, origin/destination identifiers, scheduled vs. actual times, delay minutes, distance, and cancellation flags are all 0% null and map directly onto the star schema.

---

### Next steps
1. Define the parquet schema (column selection + types) reflecting the decisions above.
2. Scaffold the Airflow ingestion DAG: monthly backfill (2018–present) → download → unzip → cast to parquet → upload to GCS.

In [16]:
BTS_DTYPES = {
    # Date / calendar
    "Year": "int16",
    "Quarter": "int8",
    "Month": "int8",
    "DayofMonth": "int8",
    "DayOfWeek": "int8",
    "FlightDate": "string",          # cast to datetime in dbt, not here — avoids tz/format assumptions baked into ingestion

    # Carrier identifiers
    "Reporting_Airline": "category",
    "DOT_ID_Reporting_Airline": "Int32",
    "IATA_CODE_Reporting_Airline": "category",
    "Tail_Number": "string",
    "Flight_Number_Reporting_Airline": "Int32",

    # Origin
    "OriginAirportID": "Int32",
    "OriginAirportSeqID": "Int32",
    "OriginCityMarketID": "Int32",
    "Origin": "category",
    "OriginCityName": "string",
    "OriginState": "category",
    "OriginStateFips": "category",
    "OriginStateName": "string",
    "OriginWac": "Int16",

    # Destination — same shape as Origin
    "DestAirportID": "Int32",
    "DestAirportSeqID": "Int32",
    "DestCityMarketID": "Int32",
    "Dest": "category",
    "DestCityName": "string",
    "DestState": "category",
    "DestStateFips": "category",
    "DestStateName": "string",
    "DestWac": "Int16",

    # Departure performance — HHMM ints kept raw; real timestamp math is dbt's job
    "CRSDepTime": "Int32",
    "DepTime": "Int32",
    "DepDelay": "float32",
    "DepDelayMinutes": "float32",
    "DepDel15": "Int8",              # nullable 0/1 flag
    "DepartureDelayGroups": "Int8",
    "DepTimeBlk": "category",
    "TaxiOut": "float32",
    "WheelsOff": "Int32",

    # Arrival performance
    "WheelsOn": "Int32",
    "TaxiIn": "float32",
    "CRSArrTime": "Int32",
    "ArrTime": "Int32",
    "ArrDelay": "float32",
    "ArrDelayMinutes": "float32",
    "ArrDel15": "Int8",
    "ArrivalDelayGroups": "Int8",
    "ArrTimeBlk": "category",

    # Cancellation / diversion (summary)
    "Cancelled": "boolean",
    "CancellationCode": "category",
    "Diverted": "boolean",

    # Duration & distance
    "CRSElapsedTime": "float32",
    "ActualElapsedTime": "float32",
    "AirTime": "float32",
    "Flights": "int8",
    "Distance": "float32",
    "DistanceGroup": "int8",

    # Delay cause breakdown — sparse by design, keep as-is
    "CarrierDelay": "float32",
    "WeatherDelay": "float32",
    "NASDelay": "float32",
    "SecurityDelay": "float32",
    "LateAircraftDelay": "float32",

    # Gate return / cancelled flight ground time
    "FirstDepTime": "Int32",
    "TotalAddGTime": "float32",
    "LongestAddGTime": "float32",

    # Diversion detail (summary)
    "DivAirportLandings": "int8",
    "DivReachedDest": "boolean",
    "DivActualElapsedTime": "float32",
    "DivArrDelay": "float32",
    "DivDistance": "float32",
}

# Div1–5 block: identical 8-field pattern, repeats for N in 1..5
for n in range(1, 6):
    BTS_DTYPES.update({
        f"Div{n}Airport": "category",
        f"Div{n}AirportID": "Int32",
        f"Div{n}AirportSeqID": "Int32",
        f"Div{n}WheelsOn": "Int32",
        f"Div{n}TotalGTime": "float32",
        f"Div{n}LongestGTime": "float32",
        f"Div{n}WheelsOff": "Int32",
        f"Div{n}TailNum": "string",
    })

In [ ]:
df["_source_file"] = source_zip_filename   # e.g. "On_Time..._2024_1.zip" — traceability
df["_ingested_at"] = pd.Timestamp.utcnow()

In [18]:
import io, zipfile, requests, pandas as pd
from collections import defaultdict

BASE = ("https://transtats.bts.gov/PREZIP/"
        "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip")

def get_columns(year, month):
    r = requests.get(BASE.format(y=year, m=month), verify=False, timeout=300)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        with z.open(name) as f:
            return list(pd.read_csv(f, nrows=0).columns)

# sample January of a few years — adjust range to whatever history you plan to ingest
samples = {y: get_columns(y, 1) for y in [2015, 2018, 2020, 2022, 2024, 2026]}

for y, cols in samples.items():
    print(f"{y}: {len(cols)} columns")

# what changed between consecutive samples
years = sorted(samples)
for a, b in zip(years, years[1:]):
    added   = set(samples[b]) - set(samples[a])
    removed = set(samples[a]) - set(samples[b])
    if added or removed:
        print(f"\n{a} → {b}")
        if added:   print(f"  + {sorted(added)}")
        if removed: print(f"  - {sorted(removed)}")

/Users/filip/Developer/us-flight-delays-pipeline/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/filip/Developer/us-flight-delays-pipeline/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/filip/Developer/us-flight-delays-pipeline/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'transtats.bts.gov'. Adding certificate verification is strongly advised. See: https://urllib3.read

2015: 110 columns
2018: 110 columns
2020: 110 columns
2022: 110 columns
2024: 110 columns
2026: 110 columns


In [19]:
type(df.columns)


pandas.Index